# Stage 01 — GPU Dual-LLM Qualitative Coding & Reconciliation (`01_gpu_coders.ipynb`)

- **Active Run ID**: `run_20261111_002`
- **Participating Agents**: Agent 07 (`src/runtime/`), Agent 02 (`src/nlp/`), Agent 03 (`src/evidence/`)

## Gate Enforcement
Full-survey qualitative coding is strictly gated on:
1. **Agent 01 Gate 1**: `data/normalized/answers_normalized.parquet` ingested and verified.
2. **Agent 02 Coding Schemas**: `src/subtopic_extraction.py` and `src/value_extraction.py` validated.


In [6]:
!python -m src.ingestion --raw-csv data/raw/survey_raw.csv --run-id run_20261111_002

Ingesting raw survey from data/raw/survey_raw.csv...
Successfully normalized 154 answer records across 11 participants.
Epistemic Status Breakdown:
  substantive: 131
  raw_blank: 10
  mixed_uncertainty_and_preference: 5
  ambiguous: 2
  explicit_no_opinion: 2
  explicit_insufficient_experience: 2
  punctuation_uninterpretable: 2
Missingness State Breakdown:
  observed_usable: 136
  missing_raw_blank: 10
  missing_non_substantive: 6
  ambiguous: 2
Text supplied: 144 / 154
Has usable preference: 136 / 154


In [7]:
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/HFG'), Path('/content')]
    for p in candidates:
        if (p / 'src').is_dir() and ((p / 'pyproject.toml').is_file() or (p / 'data').is_dir()):
            return p.resolve()
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None and Path('/content').is_dir():
    target = Path('/content/HFG')
    print(f'Repository not found locally; cloning into {target}...')
    try:
        subprocess.run(['git', 'clone', 'https://github.com/necantis/HFG.git', str(target)], check=True)
        REPO_ROOT = _find_repo_root()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Automatic git clone failed. If the repo is private, clone it manually or mount Google Drive and cd into HFG.'
        ) from exc

if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate HFG repo root from {Path.cwd()}. Clone/open inside HFG first.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'HFG repo root: {REPO_ROOT}')

import json
from src.runtime import (
    GPURuntimeConfig,
    DualCoderRuntimeManager,
    detect_hardware_profile,
)

config_path = Path('runs/run_20261111_002/gpu_runtime_config.json')
cfg = GPURuntimeConfig.load(config_path) if config_path.is_file() else GPURuntimeConfig(run_id='run_20261111_002')
mgr = DualCoderRuntimeManager(config=cfg)
preflight = mgr.run_hardware_preflight()
print('Hardware Preflight:', json.dumps(preflight, indent=2))


HFG repo root: /content/HFG
Hardware Preflight: {
  "preflight_passed": true,
  "hardware": {
    "measured_at_utc": "2026-11-13T18:39:34.228165+00:00",
    "hostname": "7d4213ec7641",
    "os_platform": "Linux 6.6.159+ (x86_64)",
    "cpu_model": "AMD EPYC 9B45",
    "cpu_logical_cores": 48,
    "system_ram_total_mb": 181130.01,
    "system_ram_available_mb": 178091.84,
    "gpu_available": true,
    "gpu_count": 1,
    "gpu_device_name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
    "gpu_total_vram_mb": 97887.0,
    "gpu_free_vram_mb": 97251.0,
    "gpu_used_vram_mb": 0.0,
    "cuda_version": "13.0",
    "driver_version": "580.82.07",
    "compute_capability": "12.0",
    "detection_method": "nvidia-smi"
  },
  "derived_budget": {
    "execution_mode": "CONCURRENT_DUAL_SERVER",
    "concurrent_feasible": true,
    "single_gpu_feasible": true,
    "gpu_memory_fraction": 0.5449,
    "coder_a_memory_fraction": 0.2945,
    "coder_b_memory_fraction": 0.2504,
    "reserved_vram_headr

In [8]:
# Check upstream Gate 1 (Agent 01) and Gate 2 (Agent 02) before authorizing full-survey coding
authorized, gate_status = mgr.verify_full_survey_coding_gates(Path.cwd())
print(f'Full Survey Coding Authorized: {authorized} ({gate_status})')
if not authorized:
    print('Skipping full-survey qualitative coding run until Gate 1 and Agent 02 schemas pass.')


Full Survey Coding Authorized: True (AUTHORIZED)
